# ITS Digital Twin: Perception Pipeline
This notebook downloads the AIC22 CityFlow dataset, extracts vehicle trajectories, projects them to 3D world coordinates, and exports the data for the Three.js frontend.

In [1]:
!pip install -q ultralytics supervision opencv-python-headless kaggle scipy numpy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.7/45.7 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.2/88.2 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 40.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 373.3/373.3 kB 37.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 66.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.3/144.3 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.3/65.3 kB 5.5 MB/s eta 0:00:00


In [ ]:
import os
from google.colab import userdata
try:
    os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
    print('Kaggle credentials loaded from Colab Secrets.')
except Exception:
    import getpass
    print('Please enter your Kaggle credentials.')
    os.environ['KAGGLE_USERNAME'] = getpass.getpass('KAGGLE_USERNAME: ')
    os.environ['KAGGLE_KEY'] = getpass.getpass('KAGGLE_KEY: ')

!kaggle datasets download -d shanthinisampath1/cityflow-multi-camera-tracking-dataset --unzip -p ./cityflow

Kaggle credentials loaded from Colab Secrets.
Dataset URL: https://www.kaggle.com/datasets/shanthinisampath1/cityflow-multi-camera-tracking-dataset
License(s): Attribution-NonCommercial 4.0 International (CC BY-NC 4.0)
100% 15.8G/15.8G [03:15<00:00, 86.8MB/s]



In [5]:
import numpy as np
import cv2

# --- 1. Manual 4-Point Ground Plane Homography ---
# Define 4 image pixel points corresponding to visible road quadrilateral on pavement:
# Top-Left: [350, 480], Top-Right: [1550, 480], Bottom-Right: [1850, 980], Bottom-Left: [100, 980]
src_pts = np.float32([[350, 480], [1550, 480], [1850, 980], [100, 980]])

# Map them to a metric 3D ground plane rectangle (X: -50m to +50m, Z: -50m to +50m)
dst_pts = np.float32([[-50, -50], [50, -50], [50, 50], [-50, 50]])

# Compute Homography
H, _ = cv2.findHomography(src_pts, dst_pts)

print('Ground-Plane Homography Matrix (H):')
print(H)

def project_to_ground(u, v, H):
    px = np.array([[[u, v]]], dtype=np.float32)
    world_pt = cv2.perspectiveTransform(px, H)[0][0]
    return float(world_pt[0]), float(world_pt[1])


Homography Matrix (H):
[[-3.33913162e+01 -2.46452663e+01 -8.15911777e+02]
 [-3.18750082e+00  4.00210599e-01  1.71861512e+02]
 [-1.65150000e-02  3.28122688e-03  1.00000000e+00]]


In [ ]:
import cv2
import json
import numpy as np
from ultralytics import YOLO
from scipy.signal import savgol_filter

model = YOLO('yolov8x.pt')
video_path = './cityflow/train/S01/c001/vdo.avi'
output_video_path = 'feed_raw.mp4'
output_json_path = 'scene_data.json'

cap = cv2.VideoCapture(video_path)
fps = int(cap.get(cv2.CAP_PROP_FPS))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_video_path, fourcc, fps, (width, height))

print(f"Processing video: {width}x{height} @ {fps}fps, total frames: {total_frames}")

results = model.track(source=video_path, persist=True, stream=True, classes=[2, 3, 5, 7])

# --- Pass 1: Extract all bounding boxes and project to ground coordinates ---
tracks = {} # track_id -> { 'frames': [], 'x': [], 'z': [], 'bbox': [], 'class_name': '' }
first_frame_img = None

frame_idx = 1
for r in results:
    frame = r.orig_img
    if first_frame_img is None:
        first_frame_img = frame.copy()
        
    if r.boxes.id is not None:
        boxes = r.boxes.xyxy.cpu().numpy()
        track_ids = r.boxes.id.cpu().numpy().astype(int)
        class_ids = r.boxes.cls.cpu().numpy().astype(int)
        
        for box, track_id, class_id in zip(boxes, track_ids, class_ids):
            x1, y1, x2, y2 = box
            class_name = model.names[class_id]
            
            # Ground contact point: bottom-center of bounding box
            u = (x1 + x2) / 2.0
            v = float(y2)
            
            px = np.array([[[u, v]]], dtype=np.float32)
            world_pt = cv2.perspectiveTransform(px, H)[0][0]
            x_world = float(world_pt[0])
            z_world = float(world_pt[1])
            
            if track_id not in tracks:
                tracks[track_id] = {'frames': [], 'x': [], 'z': [], 'bbox': [], 'class_name': class_name}
                
            tracks[track_id]['frames'].append(frame_idx)
            tracks[track_id]['x'].append(x_world)
            tracks[track_id]['z'].append(z_world)
            tracks[track_id]['bbox'].append([float(x1), float(y1), float(x2), float(y2)])
            
    out.write(frame)
    frame_idx += 1

out.release()
cap.release()

# Re-encode to H264 for web browser compatibility
!ffmpeg -y -hide_banner -loglevel error -i feed_raw.mp4 -vcodec libx264 -crf 23 feed.mp4

# --- Pass 2 & 3: Filter, Savitzky-Golay Smoothing, Speed & Yaw ---
MIN_TRACK_LENGTH = 15
frames_data = {}
for i in range(1, frame_idx):
    frames_data[str(i)] = []

world_bounds = {'minX': -50.0, 'maxX': 50.0, 'minZ': -50.0, 'maxZ': 50.0}

for track_id, data in tracks.items():
    if len(data['frames']) < MIN_TRACK_LENGTH:
        continue
        
    x_arr = np.array(data['x'])
    z_arr = np.array(data['z'])
    n_pts = len(x_arr)
    
    # Savitzky-Golay smoothing
    window_length = min(11, n_pts)
    if window_length % 2 == 0:
        window_length -= 1
    if window_length >= 3:
        x_smooth = savgol_filter(x_arr, window_length=window_length, polyorder=2)
        z_smooth = savgol_filter(z_arr, window_length=window_length, polyorder=2)
    else:
        x_smooth = x_arr
        z_smooth = z_arr
        
    yaw_arr = np.zeros(n_pts)
    speed_arr = np.zeros(n_pts)
    
    for i in range(n_pts):
        # 5-frame window for distance & speed calculation (0.5s at 10 FPS)
        idx_prev_5 = max(0, i - 5)
        dt_5 = (data['frames'][i] - data['frames'][idx_prev_5]) / fps
        dx_5 = x_smooth[i] - x_smooth[idx_prev_5]
        dz_5 = z_smooth[i] - z_smooth[idx_prev_5]
        dist_5 = np.sqrt(dx_5**2 + dz_5**2)
        
        speed = (dist_5 / dt_5 * 3.6) if dt_5 > 0 else 0.0
        speed_arr[i] = speed
        
        # 3-frame window for heading (yaw) calculation
        idx_prev_3 = max(0, i - 3)
        dx_3 = x_smooth[i] - x_smooth[idx_prev_3]
        dz_3 = z_smooth[i] - z_smooth[idx_prev_3]
        
        # If speed < 2.0 km/h, preserve previous yaw to prevent vehicle spinning
        if speed < 2.0 or (dx_3 == 0 and dz_3 == 0):
            yaw_arr[i] = yaw_arr[i - 1] if i > 0 else 0.0
        else:
            yaw_arr[i] = np.arctan2(dz_3, dx_3)
            
    # Unwrap yaw for smooth, continuous rotation without 2*pi snaps
    yaw_smooth = np.unwrap(yaw_arr)
    
    # Assemble frame data
    for i, f_idx in enumerate(data['frames']):
        frames_data[str(f_idx)].append({
            'id': int(track_id),
            'class_name': data['class_name'],
            'x': float(x_smooth[i]),
            'z': float(z_smooth[i]),
            'yaw': float(yaw_smooth[i]),
            'speed_kmh': float(speed_arr[i]),
            'bbox': data['bbox'][i]
        })

# --- Pass 4: BEV Background Orthophoto Generation ---
bev_texture_path = 'textures/road_bev.png'

# 1000x1000 BEV texture covering the -50m to +50m ground plane
dst_bev_pts = np.float32([[0, 0], [1000, 0], [1000, 1000], [0, 1000]])
H_bev = cv2.getPerspectiveTransform(src_pts, dst_bev_pts)
bev_img = cv2.warpPerspective(first_frame_img, H_bev, (1000, 1000))

cv2.imwrite('road_bev.png', bev_img)

scene_data = {
    'meta': {
        'fps': fps,
        'total_frames': frame_idx - 1,
        'world_bounds': world_bounds,
        'bev_texture': bev_texture_path
    },
    'frames': frames_data
}

with open(output_json_path, 'w') as f:
    json.dump(scene_data, f)

print(f"Export complete: feed.mp4, {output_json_path}, road_bev.png")


Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.
requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
Using Python 3.13.15 environment at: /usr
Resolved 2 packages in 259ms
Prepared 1 package in 59ms
Installed 1 package in 1ms
 + lap==0.5.13

requirements: AutoUpdate success ✅ 0.7s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


video 1/1 (frame 1/1955) /content/cityflow/train/S01/c001/vdo.avi: 384x640 2 cars, 62.2ms
video 1/1 (frame 2/1955) /content/cityflow/train/S01/c001/vdo.avi: 384x640 1 car, 62.1ms
video 1/1 (frame 3/1955) /content/cityflow/train/S01/c001/vdo.avi: 384x640 1 car, 62.1ms
video 1/1 (frame 4/1955) /content/cityflow/train/S01/c001/vdo.av